In [26]:
from typing import Annotated, TypedDict
from langchain_groq import ChatGroq
from langchain_core.messages import AnyMessage , AIMessage, BaseMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
# imp
from langgraph.types import interrupt, Command
from dotenv import load_dotenv


In [27]:
load_dotenv()

True

In [28]:
llm = ChatGroq(model="llama-3.1-8b-instant")

In [29]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage],add_messages]

In [30]:
def chat_node(state:ChatState):
    decision = interrupt({
        'type':'approval',
        'reason':'model is about to answer a user question.',
        'question': state['messages'][-1].content,
        'instruction':'Approve this Question? yes/no'
    })

    if decision['approved'] == 'no':
        return {'messages': [AIMessage(content='Not approved')]}
    else:
        res = llm.invoke(state['messages'])
        return {'messages':[res]}


In [31]:
builder = StateGraph(ChatState)

builder.add_node("chat",chat_node)

builder.add_edge(START, "chat")
builder.add_edge("chat", END)

# Checkpointer is required for interrupts
checkpointer = MemorySaver()
app = builder.compile(checkpointer=checkpointer)

In [32]:
# Create a new thread id for this conversation
config = {"configurable": {"thread_id": "1234"}}
# STEP 1: user asks a question
initiat_input = {
    "messages": [
        ("user", "Explain gradient descent in very simple terms.")
    ]
}
# Invoke the graph for the first time
result = app.invoke(initiat_input, config)



In [33]:
result

{'messages': [HumanMessage(content='Explain gradient descent in very simple terms.', additional_kwargs={}, response_metadata={}, id='e1c581be-ed7e-4a21-b69f-a9dc9e4c9fee')],
 '__interrupt__': [Interrupt(value={'type': 'approval', 'reason': 'model is about to answer a user question.', 'question': 'Explain gradient descent in very simple terms.', 'instruction': 'Approve this Question? yes/no'}, id='045bf82b65ec7b2d00f34546e1842019')]}

In [34]:
message = result['__interrupt__'][0].value
message

{'type': 'approval',
 'reason': 'model is about to answer a user question.',
 'question': 'Explain gradient descent in very simple terms.',
 'instruction': 'Approve this Question? yes/no'}

In [35]:
user_input = input(f"\n Backend message- {message} \n  Approve this question? y/n")

In [36]:
final_res = app.invoke(
    Command(resume={"approved": user_input}),
    config=config
)

In [38]:
final_res

{'messages': [HumanMessage(content='Explain gradient descent in very simple terms.', additional_kwargs={}, response_metadata={}, id='e1c581be-ed7e-4a21-b69f-a9dc9e4c9fee'),
  AIMessage(content='**Gradient Descent in Simple Terms**\n\nImagine you\'re trying to find the lowest point in a big valley. You\'re standing at the top of the valley, and you want to get to the bottom as quickly as possible.\n\n**How You\'d Normally Do It:**\n\n1. You start by going downhill in a random direction.\n2. As you walk, you look around and see that you\'re still quite high up.\n3. You adjust your direction slightly and continue walking downhill.\n4. You keep doing this, adjusting your direction based on how steep the hill is in front of you.\n5. Eventually, you get to the bottom of the valley.\n\n**Gradient Descent:**\n\nGradient descent is an algorithm that does the same thing, but with math. It\'s used to find the minimum value of a function (like the lowest point in the valley).\n\nHere\'s how it wor